# Patch GP and CNN reconstruction of raw dMRI

Both methods receive raw diffusion-weighted intensities. The short examples use one selected image patch.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Load one diffusion-weighted image

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dmri_io import load_dmri_study
from dmri_gp import patch_gp_upsample
from dmri_cnn import interpolate_cnn_patch

study = load_dmri_study(PROJECT_ROOT)
SLICE_INDEX, GRADIENT_INDEX = 19, 15
high = study.data[:, :, SLICE_INDEX, GRADIENT_INDEX]
low = high[::2, ::2]

## 3. Patch-based Gaussian process

In [ ]:
low_roi = low[36:64, 46:74]
reference_roi = high[72:128, 92:148]
gp_roi = patch_gp_upsample(
    low_roi, reference_roi.shape, patch_size=10, factor=2,
    optimize_kernel=False, show_progress=True,
)
print('GP RMSE:', float(np.sqrt(np.mean((gp_roi-reference_roi)**2))))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(reference_roi.T, cmap='gray', origin='lower'); axes[0].set_title('Reference')
axes[1].imshow(gp_roi.T, cmap='gray', origin='lower'); axes[1].set_title('Patch GP')
for ax in axes: ax.axis('off')
plt.tight_layout()

## 4. Coordinate CNN translated from the original Python/MATLAB demo

In [ ]:
CNN_EPOCHS = 20  # Change to 2000 for the original long configuration.
cnn = interpolate_cnn_patch(
    study.data, slice_index=SLICE_INDEX, gradient_index=GRADIENT_INDEX,
    low_resolution_start=(40, 50), patch_size=15, factor=2,
    epochs=CNN_EPOCHS, verbose=0,
)
print('CNN normalized MSE:', cnn['normalized_mse'])
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cnn['reference_patch'].T, cmap='gray', origin='lower'); axes[0].set_title('Reference')
axes[1].imshow(cnn['predicted_patch'].T, cmap='gray', origin='lower'); axes[1].set_title('CNN')
for ax in axes: ax.axis('off')
plt.tight_layout()